# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [4]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [5]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    !cd ECE1508_GenAI && git pull

%cd ECE1508_GenAI

remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 18 (delta 9), reused 18 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (18/18), 442.58 KiB | 13.41 MiB/s, done.
From https://github.com/WoodyChang21/ECE1508_GenAI
   067885c..ba4252a  steven     -> origin/steven
Updating 067885c..ba4252a
error: Your local changes to the following files would be overwritten by merge:
	steven/outputs/metrics.json
Please commit your changes or stash them before you merge.
Aborting
/content/ECE1508_GenAI


In [6]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

In [7]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [8]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI
plugins: typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collected 20 items                                                             

steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [  5%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [ 10%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [ 15%]
steven/tests/test_data_pipeline.py::test_build_window_shapes_and_masks PASSED [ 20%]
steven/tests/test_data_pipeline.py::test_to_patchtst_input_patch_padding_mask PASSED [ 25%]
steven/tests/test_data_pipeline.py::test_window_sampler_unique_and_within_bounds PASSED [ 30%]
steven/tests/test_data_pipeline.py::test_window_sampler_respects_split_boundary PASSED [ 35%]

## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full config.

In [9]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

04:40:26 device: cuda
04:40:26 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
04:40:26 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
04:40:30 epoch 1/20  train_loss=0.21828  val_loss=0.10680  (2.5s)
04:40:30   -> saved best checkpoint (val_loss=0.10680) to steven/outputs/patchtst_checkpoint.pt
04:40:32 epoch 2/20  train_loss=0.16714  val_loss=0.09998  (1.9s)
04:40:32   -> saved best checkpoint (val_loss=0.09998) to steven/outputs/patchtst_checkpoint.pt
04:40:34 epoch 3/20  train_loss=0.15177  val_loss=0.09655  (1.9s)
04:40:34   -> saved best checkpoint (val_loss=0.09655) to steven/outputs/patchtst_checkpoint.pt
04:40:36 epoch 4/20  train_loss=0.14239  val_loss=0.09551  (2.0s)
04:40:36   -> saved best checkpoint (val_loss=0.09551) to steven/outputs/patchtst_checkpoint.pt
04:40:38 epoch 5/20  train_loss=0.13964  val_loss=0.09426  (1.9s)
04:40:38   -> saved best checkpoint (val_loss=0.09426) to steven/outputs/patchtst_checkpoint.

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [10]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

04:41:10 device: cuda
04:41:10 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
04:41:10 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
04:41:14 epoch 1/30  beta=0.20  train_loss=0.58077 (kl=0.8095)  val_loss=0.35715 (kl=0.8000)  (2.1s)
04:41:14   -> saved best checkpoint (val_loss=0.35715) to steven/outputs/cvae_checkpoint.pt
04:41:15 epoch 2/30  beta=0.40  train_loss=0.61298 (kl=0.8001)  val_loss=0.50266 (kl=0.8000)  (1.5s)
04:41:17 epoch 3/30  beta=0.60  train_loss=0.74139 (kl=0.8000)  val_loss=0.65874 (kl=0.8000)  (1.4s)
04:41:18 epoch 4/30  beta=0.80  train_loss=0.85813 (kl=0.8001)  val_loss=0.78994 (kl=0.8000)  (1.4s)
04:41:20 epoch 5/30  beta=1.00  train_loss=0.99277 (kl=0.8000)  val_loss=0.93811 (kl=0.8000)  (1.4s)
04:41:21 epoch 6/30  beta=1.00  train_loss=0.98120 (kl=0.8000)  val_loss=0.93389 (kl=0.8000)  (1.4s)
04:41:22 epoch 7/30  beta=1.00  train_loss=0.98207 (kl=0.8000)  val_loss=0.92758 (kl=0.8000)  (1.4s)
04:41:24

## Evaluate both models on the fixed test set

In [11]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

04:42:00 device: cuda
04:42:00 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
04:42:00 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
04:42:00 evaluating on 3000 fixed test windows
04:42:01 wrote metrics to steven/outputs/metrics.json
04:42:01 overall: {
  "n_windows": 3000,
  "patchtst_reparam_mae_rmse": [
    0.09906471520662308,
    0.3766257166862488
  ],
  "cvae_reparam_mae_rmse": [
    0.1825673133134842,
    0.5133175849914551
  ],
  "patchtst_ohlc_mae_rmse": [
    2.41496778668275,
    3.569929247737977
  ],
  "cvae_ohlc_mae_rmse": [
    17.5022454512394,
    22.796515828072867
  ],
  "patchtst_volume_mae_rmse": [
    2011212.25,
    3498260.75
  ],
  "cvae_volume_mae_rmse": [
    3355692.75,
    4539899.0
  ],
  "patchtst_directional_accuracy": [
    0.4696666666666667,
    0.466,
    0.4726666666666667
  ],
  "cvae_directional_accuracy": [
    0.4723333333333333,
    0.502,
    0.5406666666666666
  ],
  "cvae_avg_samp

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [12]:
!python steven/src/update_report.py

python3: can't open file '/content/ECE1508_GenAI/steven/src/update_report.py': [Errno 2] No such file or directory


## Pull results back down

Zips `steven/outputs/` (checkpoints, metrics.json, sample_plots) and downloads it -- or just `git add`/`commit`/`push` from here if you'd rather sync back through the repo.

In [13]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

updating: steven/outputs/ (stored 0%)
updating: steven/outputs/sample_plots/ (stored 0%)
updating: steven/outputs/sample_plots/sample0_start24561_ctx70.png (deflated 13%)
updating: steven/outputs/sample_plots/sample2_start26217_ctx28.png (deflated 13%)
updating: steven/outputs/sample_plots/sample1_start26322_ctx63.png (deflated 7%)
updating: steven/outputs/sample_plots/sample2_start24588_ctx35.png (deflated 6%)
updating: steven/outputs/sample_plots/sample4_start26357_ctx14.png (deflated 6%)
updating: steven/outputs/sample_plots/sample3_start25951_ctx28.png (deflated 7%)
updating: steven/outputs/sample_plots/sample3_start25919_ctx49.png (deflated 13%)
updating: steven/outputs/sample_plots/sample0_start26248_ctx21.png (deflated 7%)
updating: steven/outputs/sample_plots/sample4_start26212_ctx49.png (deflated 13%)
updating: steven/outputs/sample_plots/sample1_start24657_ctx42.png (deflated 13%)
updating: steven/outputs/metrics.json (deflated 84%)
updating: steven/outputs/cvae_checkpoint.pt

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>